In [4]:
import tensorflow as tf
import glob
import os

from exodash.utils.production_sector import ProductionSector

ASTRO_LABELS = {
    45626007401: "j",
    1756914901: "j",
    171707907101: "e",
    39337934801: "e",
    34081054201: "e",
    10797126301: "j",
    41907645801: "j",
    13842273501: "j",
    10214547101: "e",
    34959893601: "j",
    16550244701: "e",
    30598204501: "e",
    40666958401: "e",
    27849265801: "e",
    27946260501: "e",
    972019901: "e",
    14799327701: "e",
    844471301: "e",
    33643453201: "e",
    43170146201: "j",
    18300448601: "e",
    44026043301: "e",
    24043607501: "e",
    39247608001: "e",
    4419334201: "e",
    6632618801: "e",
    11638597701: "j",
    19552641701: "e",
    13398751001: "j",
    31737242601: "e",
    35487966701: "e",
    25918291001: "e",
    41043345201: "j",
    23372747801: "j",
    21911081401: "e",
    27849683601: "e",
    1674010101: "e",
    46689506901: "j",
    20749315201: "e",
    110184411901: "j",
    22459838401: "e",
    42011477201: "e",
    2669144501: "e",
    45239177101: "e",
    19006172301: "e",
    13752049301: "j",
    15830859701: "e",
    31354903401: "e",
    11227261701: "j",
    23675560701: "j",
    20739763801: "e",
    14117924701: "e",
    14891384801: "j",
    31223056601: "e",
    28920605801: "e",
    32183240901: "e",
    24798746901: "e",
}

OUTPUT_TFRECORD = "/pdo/astronet-data/data/tfrecords/extracted/sectors_73_to_84_new_labels.tfrecord"

sectors = [73, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84]
production_sectors = []
for sector in sectors:
    production_sectors.append(ProductionSector(sector=sector, tfrecord_postfix='scatter'))

def set_int_feature(example, name, value):
    example.features.feature[name].int64_list.value[:] = [int(value)]

def inject_labels(example, astro_id):
    """Inject label columns into example based on annotation."""
    label = ASTRO_LABELS.get(astro_id)
    if label is None:
        return False  # skip unlabeled examples

    disp = {"disp_p": 0, "disp_e": 0, "disp_n": 0, "disp_j": 0}
    if label == "e":
        disp["disp_e"] = 1
    elif label == "j":
        disp["disp_j"] = 1
    elif label == "n":
        disp["disp_n"] = 1
    elif label == "p":
        disp["disp_p"] = 1

    for col, val in disp.items():
        set_int_feature(example, col, val)
    return True

def extract_tic_ids_from_production_sectors(
    production_sectors,
    astro_labels,
    output_path: str,
) -> None:
    eval_files = []
    for prod_sector in production_sectors:
        eval_files.extend(prod_sector.eval_files)

    # Strip name: prefix if present
    file_patterns = []
    for f in eval_files:
        if ":" in f:
            _, pattern = f.split(":", 1)
        else:
            pattern = f
        file_patterns.extend(glob.glob(pattern) if "*" in pattern else [pattern])

    input_files = sorted(f for f in file_patterns if os.path.isfile(f))
    print(f"Scanning {len(input_files)} tfrecord files...")

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    count = 0
    scanned = 0
    #print(input_files)
    with tf.io.TFRecordWriter(output_path) as writer:
        dataset = tf.data.TFRecordDataset(input_files)
        for raw_record in dataset:
            scanned += 1
            example = tf.train.Example()
            example.ParseFromString(raw_record.numpy())

            astro_id = int(example.features.feature['astro_id'].int64_list.value[0])
            tic_id = astro_id // 100
            if astro_id in astro_labels:
                print('inject')
                labeled = inject_labels(example, astro_id)
                if labeled:
                    writer.write(example.SerializeToString())
                    count += 1

            if scanned % 50000 == 0:
                print(f"Scanned {scanned}, matched {count}...")

    print(f"Done. Scanned {scanned} records, wrote {count} matching to: {output_path}")


extract_tic_ids_from_production_sectors(
    production_sectors=production_sectors,  # already built above
    astro_labels=ASTRO_LABELS,
    output_path=OUTPUT_TFRECORD,
)

Scanning 500 tfrecord files...
inject


2026-04-08 23:52:10.221174: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [500]
	 [[{{node Placeholder/_0}}]]


inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
inject
Done. Scanned 12839 records, wrote 88 matching to: /pdo/astronet-data/data/tfrecords/extracted/sectors_73_to_84_new_labels.tfrecord


In [7]:
import pandas as pd

spec = pd.DataFrame([
    {"astro_id": astro_id, "sample_weight": 150.0, "split": "train"}
    for astro_id in ASTRO_LABELS
])

spec.to_parquet("/pdo/users/dimond/train_spec_sectors_73_to_84.parquet", index=False)
print(spec)

        astro_id  sample_weight  split
0    45626007401           15.0  train
1     1756914901           15.0  train
2   171707907101           15.0  train
3    39337934801           15.0  train
4    34081054201           15.0  train
5    10797126301           15.0  train
6    41907645801           15.0  train
7    13842273501           15.0  train
8    10214547101           15.0  train
9    34959893601           15.0  train
10   16550244701           15.0  train
11   30598204501           15.0  train
12   40666958401           15.0  train
13   27849265801           15.0  train
14   27946260501           15.0  train
15     972019901           15.0  train
16   14799327701           15.0  train
17     844471301           15.0  train
18   33643453201           15.0  train
19   43170146201           15.0  train
20   18300448601           15.0  train
21   44026043301           15.0  train
22   24043607501           15.0  train
23   39247608001           15.0  train
24    4419334201         

In [16]:
import tensorflow as tf

TFRECORD_PATH = "/pdo/astronet-data/data/tfrecords/extracted/tic_224295942.tfrecord"
EXPECTED_LABEL_KEYS = {"disp_p", "disp_e", "disp_n", "disp_j"}

dataset = tf.data.TFRecordDataset(TFRECORD_PATH)

for i, raw_record in enumerate(dataset):
    example = tf.train.Example()
    example.ParseFromString(raw_record.numpy())
    keys = set(example.features.feature.keys())

    print(f"\n--- Record {i} ---")
    print(f"astro_id: {example.features.feature['astro_id'].int64_list.value[0]}")

    # Check label keys
    missing = EXPECTED_LABEL_KEYS - keys
    present = EXPECTED_LABEL_KEYS & keys
    print(f"Label keys present: {present}")
    print(f"Label keys MISSING: {missing if missing else 'none'}")

    # Print label values
    for k in sorted(EXPECTED_LABEL_KEYS & keys):
        val = example.features.feature[k].int64_list.value
        print(f"  {k}: {list(val)}")


--- Record 0 ---
astro_id: 22429594201
Label keys present: set()
Label keys MISSING: {'disp_p', 'disp_j', 'disp_e', 'disp_n'}


2026-04-02 23:13:46.007539: I tensorflow/core/common_runtime/executor.cc:1197] [/device:CPU:0] (DEBUG INFO) Executor start aborting (this does not indicate an error and you can ignore this message): INVALID_ARGUMENT: You must feed a value for placeholder tensor 'Placeholder/_0' with dtype string and shape [1]
	 [[{{node Placeholder/_0}}]]
